In [15]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

# Load your code.env file
env_file = r"C:\Users\Abhijeet\code.env"

with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

print(f" Loaded from: {env_file}")

ok_openai = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI : {' Loaded' if ok_openai else ' Not found'}")

 Loaded from: C:\Users\Abhijeet\code.env
OpenAI :  Loaded


In [16]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"],
               capture_output=True)

# Load from code.env
env_file = r"C:\Users\Abhijeet\code.env"
with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

# Anthropic not needed — set dummy so scraper skips quietly
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = "not-used"

ok = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI key : {' Ready' if ok else ' Not found — check code.env'}")
if ok:
    print(" Ready to run scraper")

OpenAI key :  Ready
 Ready to run scraper


In [19]:
import asyncio
import re
import sys
import json
import urllib.request
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime


# ── SENTENCE-SAFE TRUNCATION ──────────────────────────────────────────────────

def clean_to_sentences(text: str, max_sentences: int = 2) -> str:
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    good = [
        s.strip() for s in sentences
        if len(s.strip().split()) >= 6 and re.search(r'[.!?]$', s.strip())
    ]
    return " ".join(good[:max_sentences])


# ── API KEY ───────────────────────────────────────────────────────────────────

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def _get_openai_key() -> str:
    return os.environ.get("OPENAI_API_KEY", "")

def _check_key(key: str, name: str) -> bool:
    if not key or key.startswith("YOUR_") or len(key) < 20:
        print(f"  ⚠ {name} not set — add it to your .env file")
        return False
    return True


# ── HELPERS ───────────────────────────────────────────────────────────────────

async def highlight_and_click(page, selector_or_locator, description="Action"):
    try:
        target = (
            page.locator(selector_or_locator).first
            if isinstance(selector_or_locator, str)
            else selector_or_locator
        )
        if await target.is_visible(timeout=2000):
            await target.evaluate(
                "el => { el.style.border = '4px solid #00FF00';"
                " el.style.backgroundColor = 'rgba(0,255,0,0.2)'; }"
            )
            await target.click()
            return True
    except Exception:
        pass
    return False


async def handle_cookies_automatically(page):
    for selector in [
        "text=Accept All", "text=Accept",
        "button:has-text('Accept')", "button:has-text('OK')",
        "button:has-text('I Agree')", "button:has-text('Close')",
        "button:has-text('Got it')", "button:has-text('Agree')",
    ]:
        if await highlight_and_click(page, selector, "Cookie Banner"):
            break


async def smart_goto(page, url, timeout=90000):
    try:
        await page.goto(url, wait_until="networkidle", timeout=timeout)
        return True
    except Exception as e:
        print(f"  ⚠ networkidle failed ({e.__class__.__name__}), retrying…")
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout)
        await asyncio.sleep(6)
        try:
            await page.wait_for_function(
                "() => document.body && document.body.innerText.trim().length > 200",
                timeout=15000)
        except Exception:
            pass
        return True
    except Exception as e:
        try:
            await page.goto(url, wait_until="commit", timeout=timeout)
            await asyncio.sleep(8)
            return True
        except Exception:
            pass
        print(f"  ✗ Failed: {e}")
        return False


# ── KNOWN NAMES MAP (always wins over bad page titles) ────────────────────────
# Also used as the CITY LOCK — every domain here is locked to its known city.

KNOWN_NAMES_SF = {
    "lwhs.org":             ("Lick-Wilmerding High School",                     "San Francisco, CA"),
    "townschool.com":       ("Town School for Boys",                             "San Francisco, CA"),
    "urbanschool.org":      ("The Urban School of San Francisco",                "San Francisco, CA"),
    "sfuhs.org":            ("San Francisco University High School",             "San Francisco, CA"),
    "hamlin.org":           ("The Hamlin School",                                "San Francisco, CA"),
    "bayclubs.com":         ("Bay Club",                                         "San Francisco, CA"),
    "presidiogolf.com":     ("Presidio Golf Course & Academy",                   "San Francisco, CA"),
    "revolutionprep.com":   ("Revolution Prep",                                  "San Francisco, CA"),
    "crimsoneducation.org": ("Crimson Education",                                "San Francisco, CA"),
    "spcs.stanford.edu":    ("Stanford Pre-Collegiate Studies",                  "Stanford, CA"),
    "atdp.berkeley.edu":    ("UC Berkeley Academic Talent Development Program",  "Berkeley, CA"),
    "sfballet.org":         ("San Francisco Ballet School",                      "San Francisco, CA"),
    "sfcm.edu":             ("San Francisco Conservatory of Music",              "San Francisco, CA"),
    "wellsanfrancisco.com": ("Well Clinic SF",                                   "San Francisco, CA"),
    "ypt.org":              ("Young Performers Theatre",                         "San Francisco, CA"),
    "sfgirlschorus.org":    ("San Francisco Girls Chorus",                       "San Francisco, CA"),
    "linesballet.org":      ("Alonzo King LINES Ballet",                         "San Francisco, CA"),
}

# Known factual data for institutions that are hard to scrape correctly
# Format: { domain: { field: value, ... } }
KNOWN_DATA_SF = {
    # ── VERIFIED FROM OFFICIAL SOURCES ───────────────────────────────────────
    # LWHS: founded Jan 1895 as California School of Mechanical Arts; ~550 students; 67 faculty → 8:1
    "lwhs.org": {
        "Ages": "14–18", "Students": "550", "Ratio": "8:1", "Founded": "1895",
        "Type": "Independent Day School",
    },
    # Town School: founded 1939; ~400 boys; K-8 (ages 5–14)
    "townschool.com": {
        "Ages": "5–14", "Students": "400", "Founded": "1939",
        "Type": "Independent All-Boys School",
    },
    # Urban School: founded 1966; ~420 students
    "urbanschool.org": {
        "Ages": "14–18", "Students": "420", "Founded": "1966",
        "Type": "Independent Day School",
    },
    # SFUHS: established by trustees in 1973, opened Sep 1975; ~490 students; 7:1 ratio
    "sfuhs.org": {
        "Ages": "14–18", "Students": "490", "Ratio": "7:1", "Founded": "1973",
        "Type": "Independent High School",
    },
    # Hamlin: founded 1863 by Sarah Dix Hamlin; ~430 girls; K-8 (ages 4–14)
    "hamlin.org": {
        "Ages": "4–14", "Students": "430", "Founded": "1863",
        "Type": "Independent All-Girls School",
    },
    # SF Ballet School: founded 1933; ~350 students; ages 7–18
    "sfballet.org": {
        "Ages": "7–18", "Students": "350", "Founded": "1933",
        "Type": "Ballet & Dance School",
    },
    # SFCM: founded 1917; ~400 students; primarily conservatory undergrad/grad (18–30)
    "sfcm.edu": {
        "Ages": "18–30", "Students": "400", "Founded": "1917",
        "Type": "Music Conservatory",
    },
    # SF Girls Chorus: founded 1978; ~300 singers; ages 7–18
    "sfgirlschorus.org": {
        "Ages": "7–18", "Students": "300", "Founded": "1978",
        "Type": "Choral Music Organisation",
    },
    # LINES Ballet: founded 1982 by Alonzo King; ~60 BFA students; ages 16–30
    "linesballet.org": {
        "Ages": "16–30", "Students": "60", "Founded": "1982",
        "Type": "Professional Ballet Company & School",
    },
    # YPT: founded 1983 by Matilda Manning Kunin (confirmed by user); ~200 students; ages 5–18
    "ypt.org": {
        "Ages": "5–18", "Students": "200", "Founded": "1983",
        "Type": "Youth Theatre Company",
    },
    # Bay Club: founded 1977; open to all ages
    "bayclubs.com": {
        "Ages": "All ages", "Students": "N/A", "Founded": "1977",
        "Type": "Fitness & Sports Club",
    },
    # Presidio Golf: established 1895 (confirmed in scraped About text)
    "presidiogolf.com": {
        "Ages": "All ages", "Students": "N/A", "Founded": "1895",
        "Type": "Golf Course & Academy",
    },
    # Revolution Prep: founded 2002; serves grades 6–12 (ages 11–18)
    "revolutionprep.com": {
        "Ages": "11–18", "Students": "N/A", "Founded": "2002",
        "Type": "Academic Test Prep & Tutoring",
    },
    # Crimson Education: founded 2013 in New Zealand; ages 14–22
    "crimsoneducation.org": {
        "Ages": "14–22", "Students": "N/A", "Founded": "2013",
        "Type": "College Admissions Consultancy",
    },
    # SPCS Stanford: inaugurated 2012 by Provost (per scraped text); serves grades 7–12 (ages 7–17)
    "spcs.stanford.edu": {
        "Ages": "7–17", "Students": "N/A", "Founded": "2012",
        "Type": "University Pre-Collegiate Programme",
    },
    # ATDP Berkeley: founded 1977 (since 1982 in current form per scraped text — keeping 1977)
    "atdp.berkeley.edu": {
        "Ages": "6–17", "Students": "N/A", "Founded": "1977",
        "Type": "University Academic Enrichment Programme",
    },
    # Well Clinic SF: founded 2013
    "wellsanfrancisco.com": {
        "Ages": "Adults 18+", "Students": "N/A", "Founded": "2013",
        "Type": "Therapy & Wellness Clinic",
    },
}


# ── CITY DETECTION ────────────────────────────────────────────────────────────
# FIX: City is set from KNOWN_NAMES_SF domain lock BEFORE body_text is checked.
# Body-text scanning is only used as last resort, and SF-institution domains
# can never be overridden by mentions of "New York" etc in page content.

SF_NEIGHBORHOOD_HINTS = {
    "pacific heights":  "San Francisco, CA",
    "nob hill":         "San Francisco, CA",
    "russian hill":     "San Francisco, CA",
    "haight":           "San Francisco, CA",
    "mission district": "San Francisco, CA",
    "ingleside":        "San Francisco, CA",
    "the excelsior":    "San Francisco, CA",
    "soma":             "San Francisco, CA",
    "civic center":     "San Francisco, CA",
    "noe valley":       "San Francisco, CA",
    "san francisco":    "San Francisco, CA",
    "city college of san francisco": "San Francisco, CA",
    "palo alto":        "Palo Alto, CA",
    "menlo park":       "Menlo Park, CA",
    "mountain view":    "Mountain View, CA",
    "sunnyvale":        "Sunnyvale, CA",
    "cupertino":        "Cupertino, CA",
    "los altos":        "Los Altos, CA",
    "san jose":         "San Jose, CA",
    "santa clara":      "Santa Clara, CA",
    "fremont":          "Fremont, CA",
    "oakland":          "Oakland, CA",
    "berkeley":         "Berkeley, CA",
    "walnut creek":     "Walnut Creek, CA",
    "mill valley":      "Mill Valley, CA",
    "san rafael":       "San Rafael, CA",
    "sausalito":        "Sausalito, CA",
}

# These are NEVER used to set city from body text — only SF hints are used
# This prevents "new york times" or "new york city" on a page from hijacking city
BODY_TEXT_CITY_BLOCKLIST = {
    "new york", "los angeles", "miami", "boston", "seattle", "chicago",
    "london", "sydney", "dubai", "singapore", "toronto", "hong kong",
}

def detect_city_from_body(body_text: str) -> str:
    """Scan body text for SF/Bay Area neighborhood hints only. Never returns
    out-of-region cities even if mentioned on the page."""
    sample = body_text[:6000].lower()
    for hint in sorted(SF_NEIGHBORHOOD_HINTS, key=len, reverse=True):
        if hint in sample:
            return SF_NEIGHBORHOOD_HINTS[hint]
    return "San Francisco, CA"


# ── GARBAGE SITE DETECTION ────────────────────────────────────────────────────

GARBAGE_SIGNALS = [
    "hugedomains", "domain for sale", "buy this domain", "sedoparking",
    "this domain is for sale", "godaddy", "namecheap parking",
    "page not found", "account suspended", "coming soon",
    "under construction", "website coming soon",
]

def is_garbage_site(name: str, body_text: str) -> bool:
    combined = (name + " " + body_text[:500]).lower()
    return any(sig in combined for sig in GARBAGE_SIGNALS)


# ── JUNK FILTER ───────────────────────────────────────────────────────────────

JUNK_PATTERNS = [
    r"the site you are about to visit", r"page you requested no longer exists",
    r"managed independently", r"beginning of dialog window", r"escape will cancel",
    r"enquire now$", r"apply now$", r"watch the video", r"learn more$",
    r"find out more$", r"read more$", r"click here",
    r'"school" means the school', r"student.*means the child",
    r"parents.*guardians.*applying", r"please fill out the form",
    r"fill the registration form", r"step \d+ - complete",
    r"mailing list.*keep you up to date", r"joining our mailing list",
    r"to progress your application we require",
    r"please see below the list of documents",
    r"documents that are mandatory to upload",
    r"our system is limited to uploading", r"scans of multi-page documents",
    r"tourist visa.*online", r"apply for your tourist visa",
    r"cookie", r"privacy policy", r"gdpr",
    r"we will respond to your request",
    r"managing and terminating the employment contract",
    r"personal data unless \(a\) it is provided",
    r"report lost items", r"recover belongings",
    r"lorem ipsum", r"dolor sit amet",
    r"powered by", r"photography: alonzo king",
    r"by submitting this form, i authorize",
    r"i understand that consent is not a condition",
    r"accessibility statement", r"skip to main content",
    r"join our email lists",
    r"single.elimination system",
    r"not sponsored by.*affiliated",
    r"over 1\.25 million", r"million visitor",
    # Performance field junk — content that describes nav/UI not the school
    r"^where students & faculty feed their curiosity",
    r"^our bustling, modern campus across the street",
    r"^the center is home to student leadership",
    r"^the oldest building on campus",
    r"is nestled in the ingleside neighborhood",
    r"^each of the shared spaces are used by all",
    r"^our sunlit lobby serves as",
    r"^town's beloved campus is a space",
]

def is_junk(text: str) -> bool:
    if not text:
        return True
    tl = text.lower().strip()
    return any(re.search(p, tl) for p in JUNK_PATTERNS)


# ── IMAGE HARVESTING ──────────────────────────────────────────────────────────

SKIP_IMG = [
    "logo", "icon", "svg", "1x1", "pixel", "spinner", "placeholder",
    "blank", "transparent", "avatar", "flag", "arrow", "bullet",
    "check", "star", "rating", "rs/facebook", "rs/linkedin", "rs/youtube",
    "loupe", "panier", "menu", "profil", "default_image", "black-beacon",
    "weather/128", "subtle-pattern",
]
EXTS_IMG = [".jpg", ".jpeg", ".png", ".webp"]

async def harvest_images(pg, base_url: str, image_set: set):
    try:
        await pg.evaluate("""async () => {
            await new Promise(resolve => {
                let total = 0;
                const dist = 400;
                const timer = setInterval(() => {
                    window.scrollBy(0, dist);
                    total += dist;
                    if (total >= document.body.scrollHeight) {
                        clearInterval(timer);
                        window.scrollTo(0, 0);
                        resolve();
                    }
                }, 80);
            });
        }""")
        await asyncio.sleep(1)
    except Exception:
        pass

    try:
        for img in await pg.query_selector_all("img"):
            for attr in ["src", "data-src", "data-lazy-src", "data-original",
                         "data-image", "data-bg", "data-lazy", "data-srcset"]:
                c = await img.get_attribute(attr)
                if c:
                    src = c.strip().split(" ")[0].split(",")[0].strip()
                    full = urljoin(base_url, src)
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)
                            and len(full) > 25):
                        image_set.add(full)
            srcset = await img.get_attribute("srcset")
            if srcset:
                parts = [p.strip().split(" ")[0] for p in srcset.split(",") if p.strip()]
                if parts:
                    full = urljoin(base_url, parts[-1])
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)):
                        image_set.add(full)

        for sel in ['meta[property="og:image"]', 'meta[name="twitter:image"]',
                    'meta[property="og:image:url"]']:
            try:
                for og in await pg.evaluate(
                    f"() => Array.from(document.querySelectorAll('{sel}')).map(m=>m.content)"
                ):
                    if og and og.strip() and not any(s in og.lower() for s in SKIP_IMG):
                        image_set.add(og.strip())
            except Exception:
                pass

        bg_urls = await pg.evaluate("""() => {
            const hits = [];
            const sels = [
                '[style*="background-image"]','[class*="hero"]','[class*="banner"]',
                '[class*="slider"]','[class*="slide"]','[class*="bg-image"]',
                '[class*="cover"]','section','header'
            ];
            sels.forEach(sel => {
                try {
                    document.querySelectorAll(sel).forEach(el => {
                        const s = el.style.backgroundImage ||
                                  getComputedStyle(el).backgroundImage || '';
                        const m = s.match(/url\\(["']?([^"')]+)["']?\\)/);
                        if (m && m[1]) hits.push(m[1]);
                    });
                } catch(e) {}
            });
            return [...new Set(hits)];
        }""")
        for bg in bg_urls:
            if bg:
                full = urljoin(base_url, bg.strip())
                if (any(e in full.lower() for e in EXTS_IMG)
                        and not any(s in full.lower() for s in SKIP_IMG)):
                    image_set.add(full)
    except Exception:
        pass


# ── QUOTE SCRAPER ─────────────────────────────────────────────────────────────

def _word_count(text):
    return len(text.split())

def scrape_quote_from_soup(soup):
    SKIP_PHRASES = [
        "cookie", "menu", "nav", "©", "privacy", "terms",
        "subscribe", "sign up", "click", "read more", "javascript",
        "watch the video", "watch video", "listen to", "testimonial",
        "share", "follow us", "contact us", "apply now", "find out more",
        "learn more", "get in touch", "download", "register", "book a",
        "visit us", "means the child", "parents/guardians", "dialog window",
        "escape will cancel", "site you are about to visit",
        "no longer exists", "managed independently",
        "lorem ipsum", "photography:", "by submitting",
        "one of the best parts of my job",   # head-of-school letter opener
        "we're glad you have found",          # generic welcome
        "i'm excited to share",
    ]
    selectors = [
        "blockquote", "q",
        "[class*='quote']", "[class*='pullquote']", "[class*='callout']",
        "[class*='hero'] p", "[class*='banner'] p",
        "[class*='mission'] p", "[class*='vision'] p",
        "figcaption",
    ]
    candidates = []
    for sel in selectors:
        for el in soup.select(sel):
            text = re.sub(r'\s+', ' ', el.get_text()).strip()
            text = text.strip('\u201c\u201d\u2018\u2019"\'')
            wc = _word_count(text)
            if wc < 8 or wc > 35:
                continue
            if any(x in text.lower() for x in SKIP_PHRASES):
                continue
            if is_junk(text):
                continue
            if not re.search(r'[.!?]$', text):
                continue
            candidates.append(text)
    if not candidates:
        return ""
    candidates.sort(key=lambda t: _word_count(t))
    return candidates[0]


# ── AI QUOTE ──────────────────────────────────────────────────────────────────

def generate_ai_quote(school_name, school_type, city, about_text):
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return f"Excellence, integrity, and curiosity — the pillars of {school_name}."
    prompt = (
        f"Write ONE short inspirational quote (1 sentence, 10–20 words) "
        f"that reflects the philosophy of {school_name}, a {school_type} in {city}. "
        f"Context: {about_text[:300]}. "
        f"Sound like the organisation's mission statement. "
        f"Return ONLY the quote text — no quotation marks, no attribution, no explanation."
    )
    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 60, "temperature": 0.7,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data  = json.loads(resp.read())
            quote = data["choices"][0]["message"]["content"].strip().strip('"\'')
            words = quote.split()
            if len(words) > 25:
                quote = " ".join(words[:25])
            print(f"  ✓ AI quote generated for {school_name}")
            return quote
    except Exception as e:
        print(f"  ⚠ AI quote failed: {e}")
    return f"Excellence, integrity, and curiosity — the pillars of {school_name}."


# ── SHORT KEYWORDS MAP for WP blocks ──────────────────────────────────────────

PERF_SHORT = {
    "Coaching Credentials":   "Expert Staff",
    "Student Wellbeing":      "Student Care",
    "Academic Integration":   "Curriculum",
    "Competitive Pathway":    "Achievements",
    "Facilities & Resources": "Facilities",
    "Ongoing Accountability": "Progress",
}


# ── OPENAI FORCE-FILL ALL PERFORMANCE FIELDS ──────────────────────────────────

def openai_force_fill_performance(results: dict) -> dict:
    """
    Force-fills ALL 6 performance fields using OpenAI.
    Any field that is blank, N/A, too short (<12 words), or junk
    gets a unique, substantive 2-sentence AI-generated response.
    NEVER overwrites a genuinely good scraped value.
    """
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    name        = results.get("Name", "this organisation")
    entity_type = results.get("Type", "Independent School")
    city        = results.get("City", "San Francisco, CA")
    about       = results.get("About", "")
    philosophy  = results.get("Philosophy", "")

    ctx_parts = [f"Name: {name}", f"Type: {entity_type}", f"City: {city}"]
    if about:
        ctx_parts.append(f"About: {about}")
    if philosophy:
        ctx_parts.append(f"Philosophy: {philosophy}")
    for k in ["Founded", "Ages", "Students", "Fees", "Outcomes", "Admissions"]:
        v = results.get(k, "")
        if v and v not in ("N/A", "See website", ""):
            ctx_parts.append(f"{k}: {v}")
    context = "\n".join(ctx_parts)

    PERF_PROMPTS = {
        "Coaching Credentials": (
            f"2 sentences (≤35 words total) about the qualifications, expertise, or professional "
            f"background of staff/coaches/teachers at {name}, a {entity_type} in {city}. "
            f"Mention certifications, experience years, or teaching specialisms. Be specific."
        ),
        "Student Wellbeing": (
            f"2 sentences (≤35 words total) about how {name} supports the mental, emotional, "
            f"or physical wellbeing of students. Mention counselling, advisory systems, "
            f"community building, or pastoral programmes. Be specific."
        ),
        "Academic Integration": (
            f"2 sentences (≤35 words total) about the curriculum, programmes, or learning approach "
            f"at {name}. Mention specific courses, teaching methods, or qualifications. Be specific."
        ),
        "Competitive Pathway": (
            f"2 sentences (≤35 words total) about the results, achievements, university "
            f"destinations, or career pathways of graduates from {name}. "
            f"Mention college admissions outcomes or competition awards. Be specific."
        ),
        "Facilities & Resources": (
            f"2 sentences (≤35 words total) about the physical facilities, campus spaces, "
            f"or resources available at {name}. "
            f"Mention specific amenities — studios, labs, theaters, pools, courts. Be specific."
        ),
        "Ongoing Accountability": (
            f"2 sentences (≤35 words total) about how {name} tracks student progress, "
            f"provides feedback, or reports outcomes. "
            f"Mention progress reports, parent communication, or assessment systems. Be specific."
        ),
    }

    def needs_fill(val: str) -> bool:
        if not val or val.strip() in ("N/A", "", "N/a", "See website"):
            return True
        clean = re.sub(r'\s+', ' ', val).strip()
        if len(clean.split()) < 12:
            return True
        junk_phrases = [
            "cookie", "privacy policy", "gdpr", "javascript",
            "lorem ipsum", "we will respond to your request",
            "personal data unless", "report lost items",
            "click here", "read more", "learn more", "find out more",
            "by submitting this form",
            "where students & faculty feed their curiosity",
            "our bustling, modern campus across the street",
            "the center is home to student leadership",
            "the oldest building on campus",
            "each of the shared spaces are used by all",
            "our sunlit lobby serves as",
            "town's beloved campus is a space",
            "is nestled in the ingleside",
        ]
        cl = clean.lower()
        return any(p in cl for p in junk_phrases)

    missing = {
        metric: prompt
        for metric, prompt in PERF_PROMPTS.items()
        if needs_fill(results.get("Performance", {}).get(metric, ""))
    }

    if not missing:
        print("  ✓ All performance fields complete — no AI fill needed.")
        return results

    print(f"  → AI force-filling {len(missing)} performance field(s): {', '.join(missing.keys())}")

    field_lines = "\n".join(f'  "{k}": {v}' for k, v in missing.items())

    prompt = f"""You are an expert on San Francisco Bay Area educational institutions, arts organisations, wellness centres, and sports academies.

KNOWN DATA about this institution:
{context}

Generate UNIQUE, FACTUAL, SPECIFIC content for ONLY these fields.
Each field must:
- Be exactly 2 sentences, ≤35 words total
- Describe a DIFFERENT aspect from the other fields
- Sound professional and informative (not generic)
- Use your knowledge of {name} in {city} to be accurate
- Never repeat facts from another field

Reply ONLY with valid JSON, keys exactly as shown below. No markdown, no extra keys:

{field_lines}

Return format: {{"field name": "2 sentence content here.", ...}}"""

    payload = json.dumps({
        "model": "gpt-4o",
        "max_tokens": 1200,
        "temperature": 0.4,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=40) as resp:
            data   = json.loads(resp.read())
            raw    = data["choices"][0]["message"]["content"].strip()
            raw    = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw    = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            seen_vals   = set()
            for metric, val in filled.items():
                if not val or val.strip() in ("N/A", ""):
                    continue
                key_short = re.sub(r'\s+', ' ', val.strip().lower())[:100]
                if key_short in seen_vals:
                    continue
                seen_vals.add(key_short)
                if metric in PERF_PROMPTS and needs_fill(
                    results["Performance"].get(metric, "")
                ):
                    results["Performance"][metric] = str(val).strip()
                    filled_keys.append(metric)

            if filled_keys:
                print(f"  ✓ AI filled performance: {', '.join(filled_keys)}")
            else:
                print("  ✓ AI performance: nothing new added.")

    except json.JSONDecodeError as e:
        print(f"  ⚠ AI performance JSON error: {e}")
    except Exception as e:
        print(f"  ⚠ AI performance fill failed: {e}")

    return results


# ── OPENAI MAIN FIELDS GAP-FILL ───────────────────────────────────────────────

def openai_fill_missing_fields(results: dict) -> dict:
    """Fill main fields (Founded, Ages, Students, Ratio, Fees, About, etc.)"""
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    entity_type = results.get("Type", "Independent School")
    is_school   = any(w in entity_type.lower()
                      for w in ["school", "academy", "college", "education",
                                "conservatory", "conservatoire", "programme"])
    name = results.get("Name", "this organisation")

    FILL_FIELDS = {
        "Founded":    "Year this organisation was founded (just the 4-digit year). If unknown reply N/A.",
        "Ages":       ("Student age range (e.g. '5–18'). If unknown reply N/A." if is_school
                       else "Age range of members/clients served (e.g. 'Adults 18+', 'All ages'). Never reply N/A."),
        "Students":   ("Approximate enrolled students (e.g. '800'). If unknown reply N/A." if is_school
                       else "Approximate number of members/clients/visitors annually. If unknown reply N/A."),
        "Ratio":      "Staff-to-student or instructor-to-client ratio (e.g. '1:8'). If unknown reply N/A.",
        "Fees":       "Annual tuition or membership cost in USD. SF private school tuition typically $35,000–$65,000/year. If unknown reply 'See website'.",
        "About":      f"2 sentences (30–45 words total) describing {name}'s core identity, mission, and what makes it distinctive.",
        "Philosophy": f"2 sentences (30–45 words total) on {name}'s specific teaching, coaching, or wellness approach — what do they believe and how do they do it.",
        "Outcomes":   f"2 sentences (30–45 words total) on concrete outcomes {name} achieves — university placements, awards, graduate careers, or member results.",
        "Admissions": f"2 sentences (30–45 words total) on how to apply or enrol at {name} — process, timeline, or requirements.",
        "Tagline":    f"One compelling tagline (≤20 words) for {name}.",
    }

    ctx_lines = []
    for k in ["Name", "Type", "City", "URL", "Tagline", "About", "Philosophy",
              "Outcomes", "Admissions", "Founded", "Ages", "Students", "Ratio", "Fees"]:
        v = results.get(k, "")
        if v:
            ctx_lines.append(f"{k}: {v}")
    context = "\n".join(ctx_lines) or f"URL: {results.get('URL', '')}"

    def _needs(val):
        if not val or str(val).strip() in ("N/A", "See website", "Rolling admissions",
                                            "Year-round", "", "N/a"):
            return True
        # Also re-fill if content is too short (< 20 words for paragraph fields)
        return False

    def _needs_para(field, val):
        """Paragraph fields need refill if under 20 words."""
        if _needs(val):
            return True
        if field in ("About", "Philosophy", "Outcomes", "Admissions"):
            if len(str(val).split()) < 20:
                return True
        return False

    missing = {}
    for f, p in FILL_FIELDS.items():
        v = results.get(f, "")
        if f in ("About", "Philosophy", "Outcomes", "Admissions"):
            if _needs_para(f, v):
                missing[f] = p
        elif _needs(v):
            missing[f] = p

    if not missing:
        return results

    print(f"  → OpenAI filling {len(missing)} main field(s): {', '.join(missing.keys())}")

    field_instructions = "\n".join(f'  "{k}": {v}' for k, v in missing.items())
    prompt = f"""You are a San Francisco and Bay Area education and culture expert.
Use the known data AND your knowledge to fill missing fields accurately for {name}.

KNOWN DATA:
{context}

Fill ONLY these fields (reply with a JSON object, keys exactly as shown):
{field_instructions}

Rules:
- Use factual knowledge of this specific San Francisco / Bay Area institution.
- Fees: use $ and per year. SF private school range typically $35,000–$65,000/year.
- Ages: always provide a range even for non-schools (e.g. 'Adults 18+', 'All ages', '5–18').
- Paragraph fields (About, Philosophy, Outcomes, Admissions): must be 2 full sentences, 30–45 words total. Never write fewer than 25 words.
- Each paragraph field must say something specific and substantive — not generic filler.
- Do NOT add commentary, markdown fences, or extra keys.
- Reply with valid JSON only."""

    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 1800, "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data   = json.loads(resp.read())
            raw    = data["choices"][0]["message"]["content"].strip()
            raw    = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw    = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            for fkey, val in filled.items():
                if not val or str(val).strip() in ("N/A", "See website", ""):
                    continue
                if fkey in FILL_FIELDS:
                    v = results.get(fkey, "")
                    if fkey in ("About", "Philosophy", "Outcomes", "Admissions"):
                        if _needs_para(fkey, v):
                            results[fkey] = str(val).strip()
                            filled_keys.append(fkey)
                    elif _needs(v):
                        results[fkey] = str(val).strip()
                        filled_keys.append(fkey)

            if filled_keys:
                print(f"  ✓ OpenAI filled: {', '.join(filled_keys)}")
            else:
                print("  ✓ OpenAI: nothing new to fill.")
    except Exception as e:
        print(f"  ⚠ OpenAI main fill failed: {e}")

    return results


# ── WP BLOCKS GENERATOR ───────────────────────────────────────────────────────

def generate_wp_blocks(r):
    school_name  = r.get("Name", "")
    description  = r.get("Tagline", "")
    age          = r.get("Ages", "")
    students     = r.get("Students", "")
    ratio        = r.get("Ratio", "")
    founded      = r.get("Founded", "")
    school_type  = r.get("Type", "")
    city         = r.get("City", "")
    annual_fee   = r.get("Fees", "")
    about        = r.get("About", "")
    philosophy   = r.get("Philosophy", "")
    outcomes     = r.get("Outcomes", "")
    quote        = r.get("Quote", "")
    website_url  = r.get("URL", "")
    admissions   = r.get("Admissions", "")
    images       = r.get("Images", [])
    perf         = r.get("Performance", {})
    enquiry_open = r.get("EnquiryOpen", "Year-round")
    app_deadline = r.get("AppDeadline", "Rolling admissions")

    website_display = website_url.replace("https://", "").replace("http://", "").rstrip("/")
    city_slug  = city.lower().replace(", ", "-").replace(" ", "-").replace(",", "")
    city_short = city.split(",")[0].strip()

    imgs = [img for img in images if img]
    while len(imgs) < 6:
        imgs.append(imgs[0] if imgs else "")
    img0, img1, img2, img3, img4, img5 = imgs[0], imgs[1], imgs[2], imgs[3], imgs[4], imgs[5]

    coaching       = perf.get("Coaching Credentials", "")
    wellbeing      = perf.get("Student Wellbeing", "")
    academic       = perf.get("Academic Integration", "")
    competitive    = perf.get("Competitive Pathway", "")
    facilities     = perf.get("Facilities & Resources", "")
    accountability = perf.get("Ongoing Accountability", "")

    kw_coaching       = PERF_SHORT["Coaching Credentials"]
    kw_wellbeing      = PERF_SHORT["Student Wellbeing"]
    kw_academic       = PERF_SHORT["Academic Integration"]
    kw_competitive    = PERF_SHORT["Competitive Pathway"]
    kw_facilities     = PERF_SHORT["Facilities & Resources"]
    kw_accountability = PERF_SHORT["Ongoing Accountability"]

    wp = f"""<!-- wp:image {{"id":440,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img0}" alt="{school_name}" class="wp-image-440"/></figure>
<!-- /wp:image -->

<!-- wp:paragraph -->
<p><a href="elite-home.html">Elite</a> &#8250; <a href="elite-city-{city_slug}.html">{city_short}</a> &#8250; <a href="elite-program-academic.html">Elite Academic Programs</a></p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Kidrovia Elite — Listed</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Elite Academic Programs</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":1}} -->
<h1 class="wp-block-heading"><em>{school_name}</em></h1>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{description}</p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{age}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{students}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{ratio}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{founded}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column {{"width":"66.66%"}} -->
<div class="wp-block-column" style="flex-basis:66.66%">
<!-- wp:heading --><h2 class="wp-block-heading">About <em>{school_name}</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{about}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column {{"width":"33.33%"}} -->
<div class="wp-block-column" style="flex-basis:33.33%">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Institution Details</h5><!-- /wp:heading -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Type</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">City</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{school_type}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{age}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{students}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{ratio}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{founded}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{city}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{annual_fee}</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Contact &amp; Enquiry</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading"><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display} &#8594;</a></h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">Quote</h5>
<!-- /wp:heading -->
<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Philosophy</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">How they <em>teach</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{philosophy}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Outcomes</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Where students <em>go</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{outcomes}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img1}" alt="{school_name} campus"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img2}" alt="{school_name} students"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img3}" alt="{school_name} activities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img4}" alt="{school_name} facilities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img5}" alt="{school_name} community"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":4}} -->
<h4 class="wp-block-heading">Our Assessment</h4>
<!-- /wp:heading -->
<!-- wp:heading -->
<h2 class="wp-block-heading"><strong>How {school_name} performs</strong></h2>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_coaching}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Coaching Credentials</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{coaching}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_wellbeing}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Student Wellbeing</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{wellbeing}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_academic}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Academic Integration</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{academic}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_competitive}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Competitive Pathway</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{competitive}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_facilities}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Facilities &amp; Resources</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{facilities}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_accountability}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Ongoing Accountability</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{accountability}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading --><h2 class="wp-block-heading">Admissions &amp;<br><em>How to Apply</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{admissions}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Enquiries Open</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{enquiry_open}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Application Deadline</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{app_deadline}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{annual_fee}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Location</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{city}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Website</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display}</a></p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->
"""
    return wp


# ── MAIN SCRAPER ──────────────────────────────────────────────────────────────

async def extract_school_data(url):
    async with async_playwright() as p:
        import platform
        headless = os.environ.get(
            "HEADLESS",
            "true" if platform.system() == "Linux" else "false"
        ).lower() == "true"
        browser = await p.chromium.launch(headless=headless, slow_mo=0 if headless else 200)
        context = await browser.new_context(viewport={"width": 1280, "height": 800})
        page    = await context.new_page()

        results = {
            "Name": "", "Tagline": "", "Founded": "", "City": "",
            "Ages": "", "Students": "", "Ratio": "", "Fees": "",
            "Type": "Independent School",
            "About": "", "Philosophy": "", "Outcomes": "",
            "Admissions": "", "Quote": "",
            "EnquiryOpen": "Year-round",
            "AppDeadline": "Rolling admissions",
            "Performance": {}, "URL": url, "Images": [],
        }

        global_text_pool = []

        # ── STEP 0: Lock name, city, and known data from domain map ──────────
        domain     = url.split("/")[2].replace("www.", "")
        known_entry = next(
            ((k, v) for k, v in KNOWN_NAMES_SF.items() if k in domain), None
        )
        known_name = None
        if known_entry:
            k_domain, (k_name, k_city) = known_entry
            known_name         = k_name
            results["Name"]    = k_name
            results["City"]    = k_city          # CITY LOCK — cannot be overridden
            # Apply any hardcoded factual data
            hd = KNOWN_DATA_SF.get(k_domain, {})
            for field, val in hd.items():
                results[field] = val
            print(f"  ✓ Known institution: {k_name} | City locked: {k_city}")

        FALLBACKS = {
            "Founded": "", "Ages": "", "Students": "", "Ratio": "N/A", "Fees": "See website",
        }

        fee_patterns = [
            r'\$\s*([\d,]+)(?:\.\d{2})?\s*per\s*(?:year|annum|semester|term)',
            r'tuition\s*(?:fee)?s?\s*(?:of|is|are|:)?\s*\$\s*([\d,]+)',
            r'annual\s*(?:tuition|fee)s?\s*(?:of|is|:)?\s*\$\s*([\d,]+)',
            r'fee[s]?\s*(?:of|is|are|:)?\s*\$\s*([\d,]+)',
        ]

        def extract_fee_validated(text: str) -> str:
            for pat in fee_patterns:
                for m in re.finditer(pat, text, re.I):
                    try:
                        num = int(m.group(1).replace(',', ''))
                        if 1000 <= num <= 100000:
                            return m.group(0).strip()
                    except (IndexError, ValueError):
                        pass
            return ""

        ratio_patterns = [
            r'(\d+\s*:\s*\d+)\s*(?:student[- ]to[- ](?:faculty|teacher)|teacher[- ]to[- ]student|ratio)',
            r'ratio\s*(?:of\s*)?(\d+\s*:\s*\d+)',
            r'(\d+\s*:\s*\d+)\s*ratio',
            r'1\s*(?:teacher|member of staff|coach|instructor)\s+(?:to|per|:)\s+(\d+)\s*(?:students?|pupils?|learners?)',
            r'(\d+)\s*(?:students?|pupils?|learners?)\s+(?:per|to)\s+(?:every\s+)?(?:1\s+)?(?:teacher|tutor|coach|instructor)',
        ]

        age_patterns = [
            r'students?\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'children\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'ages?\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'from\s+age\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'(\d+)\s*(?:to|[\u2013\-])\s*(\d+)\s*years?\s*(?:old)',
            r'trains?\s+students?\s+ages?\s+(\d+)[–\-](\d+)',
            r'for\s+(?:students?|children|youth|dancers?|singers?)\s+ages?\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'students\s+in\s+grades?\s+(\d+)[–\-](\d+)',
        ]

        student_patterns = [
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:pupils?|students?|boys|girls|young people|young women|learners?|scholars?)',
            r'(?:approximately|around|about|some|over|more than)\s+(\d[\d,]+)\s*(?:pupils?|students?|girls?|learners?)',
            r'school\s+of\s+(?:approximately|around|about)?\s*(\d[\d,]+)',
            r'enroll(?:ment|ment|ing)\s+(?:of\s+)?(?:approximately|about)?\s*(\d[\d,]+)',
            r'community\s+of\s+(?:over\s+)?(\d[\d,]+)\s*(?:students?|learners?|members?)',
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:student|pupil|learner|child|member)',
        ]

        print(f"Connecting to: {url}...")
        loaded = await smart_goto(page, url)
        if not loaded:
            print(f"  ✗ Could not load {url}, skipping.")
            await browser.close()
            return None

        try:
            await handle_cookies_automatically(page)

            og_site   = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:site_name\"]'); return m ? m.content : ''; }")
            og_title  = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:title\"]'); return m ? m.content : ''; }")
            raw_title = await page.title()

            REJECT_NAMES   = {"home","welcome","homepage","index","main","","welcome to","about","contact"}
            REJECT_STARTS  = ("welcome to ", "home -", "home |",
                              "sf therapy", "best child", "top fitness")
            REJECT_PHRASES = [
                "personalization", "advertising effectiveness",
                "pixelcommerce", "wix.com", "shopify", "lorem ipsum",
            ]

            def clean_name(candidate: str) -> str:
                parts = re.split(r'\s*[|—\-]\s*', candidate)
                parts = [p.strip() for p in parts if p.strip()]
                GENERIC = {"home","welcome","index","main","page","site","portal","homepage","about",""}
                n = parts[0] if parts else candidate.strip()
                if n.lower() in GENERIC and len(parts) > 1:
                    n = parts[-1].strip()
                for prefix in ("welcome to ",):
                    if n.lower().startswith(prefix):
                        n = n[len(prefix):].strip()
                return n

            # Known name always wins
            if not results["Name"]:
                for candidate in [og_site, og_title, raw_title]:
                    name = clean_name(candidate)
                    if (name
                            and name.lower() not in REJECT_NAMES
                            and not any(name.lower().startswith(p) for p in REJECT_STARTS)
                            and not any(bad in name.lower() for bad in REJECT_PHRASES)
                            and len(name.split()) <= 8):
                        results["Name"] = name
                        break
                if not results["Name"]:
                    results["Name"] = clean_name(raw_title)

            body_text = await page.inner_text("body")

            if len(body_text.strip()) < 300:
                print("  ⚠ Page body too short — scrolling to trigger JS render…")
                try:
                    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await asyncio.sleep(3)
                    await page.evaluate("window.scrollTo(0, 0)")
                    await asyncio.sleep(2)
                    body_text = await page.inner_text("body")
                except Exception:
                    pass

            # Only set city from body if not already locked by KNOWN_NAMES_SF
            if not results["City"]:
                results["City"] = detect_city_from_body(body_text)

            if is_garbage_site(results["Name"], body_text):
                print(f"  ✗ Garbage/parked site detected — skipping {url}")
                await browser.close()
                return None

            def extract(pattern):
                m = re.search(pattern, body_text, re.IGNORECASE)
                return m.group(1).strip() if m else ""

            current_year = datetime.now().year

            def extract_founded_from_text(text, existing_candidates=None):
                STRONG_KWS  = ["founded in", "established in", "was founded", "founding year",
                                "founded by", "charter", "incorporated in", "opened in",
                                "opened its doors", "since", "est."]
                WEAK_KWS    = ["founded", "established", "foundation", "history", "began"]
                SKIP_WORDS  = ["copyright", "©", "reserved", "updated", "class of", "alumni"]
                FOREIGN_CTX = ["in the uk", "in england", "in derbyshire", "in scotland",
                                "in wales", "in ireland", "parent school", "sister school"]
                cands = existing_candidates if existing_candidates is not None else []
                clean = re.sub(r'\s+', ' ', text.lower())
                clean = re.sub(r'(copyright|©).*?\d{4}', '', clean)
                for sent in re.split(r'[.!?\n]', clean):
                    if any(w in sent for w in SKIP_WORDS):
                        continue
                    has_strong = any(k in sent for k in STRONG_KWS)
                    has_weak   = any(k in sent for k in WEAK_KWS)
                    if not (has_strong or has_weak):
                        continue
                    is_foreign = any(ctx in sent for ctx in FOREIGN_CTX)
                    for y in re.findall(r'\b(1[6-9]\d{2}|20[012]\d)\b', sent):
                        year = int(y)
                        if not (1600 <= year <= current_year - 1):
                            continue
                        score = 3 if has_strong else 1
                        if year < 1900:
                            score -= 2          # light penalty, don't fully reject old schools
                        if is_foreign:
                            score -= 3
                        if year >= current_year - 2:
                            score -= 3
                        cands.append((score, year))
                return cands

            # Only scrape Founded if not already set from KNOWN_DATA_SF
            if not results.get("Founded"):
                founded_candidates = extract_founded_from_text(body_text)
                if founded_candidates:
                    founded_candidates.sort(key=lambda x: (-x[0], x[1]))
                    results["Founded"] = str(founded_candidates[0][1])

            # Ages — only scrape if not already set from KNOWN_DATA_SF
            if not results.get("Ages"):
                age_candidates = []
                for pat in age_patterns:
                    for m in re.finditer(pat, body_text, re.I):
                        try:
                            g = m.groups()
                            if len(g) >= 2 and g[0] and g[1]:
                                lo, hi = int(g[0]), int(g[1])
                                if lo < hi and lo >= 2 and hi <= 25:
                                    age_candidates.append((lo, hi))
                        except Exception:
                            pass
                if age_candidates:
                    best = max(age_candidates, key=lambda x: x[1] - x[0])
                    results["Ages"] = f"{best[0]}–{best[1]}"

                if not results["Ages"]:
                    AGE_MAP = {
                        "toddler": (1, 3), "nursery": (2, 4),
                        "pre-k": (4, 5), "prek": (4, 5), "preschool": (3, 5),
                        "transitional kindergarten": (4, 5), "kindergarten": (5, 6),
                        "lower school": (5, 11), "elementary": (5, 11),
                        "grade 1": (6, 7), "grade 5": (10, 11),
                        "grade 6": (11, 12), "middle school": (11, 14),
                        "grade 8": (13, 14), "grade 9": (14, 15),
                        "upper school": (14, 18), "high school": (14, 18),
                        "grade 12": (17, 18), "sixth form": (16, 18),
                    }
                    tl = body_text.lower()
                    fy = [v for k, v in AGE_MAP.items() if k in tl]
                    results["Ages"] = f"{min(x[0] for x in fy)}–{max(x[1] for x in fy)}" if fy else ""

            # Students — only scrape if not already set from KNOWN_DATA_SF
            if not results.get("Students"):
                for pat in student_patterns:
                    for m in re.finditer(pat, body_text, re.I):
                        num_str = re.sub(r'[\s,]', '', m.group(1))
                        try:
                            num = int(num_str)
                            if not (50 <= num <= 10000):
                                continue
                            if 2020 <= num <= 2030:
                                continue
                            ctx = body_text[max(0, m.start()-8):m.end()+8]
                            if '%' in ctx or '.' in m.group(1):
                                continue
                            results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                            break
                        except ValueError:
                            pass
                    if results["Students"]:
                        break

            # Ratio
            if not results.get("Ratio"):
                for pat in ratio_patterns:
                    m = re.search(pat, body_text, re.I)
                    if m:
                        results["Ratio"] = m.group(1).replace(" ", "")
                        break

            results["Fees"] = results.get("Fees") or extract_fee_validated(body_text)

            # Type — only set if not already locked from KNOWN_DATA_SF
            if not results.get("Type") or results["Type"] == "Independent School":
                tl    = body_text.lower()
                url_l = url.lower()
                if "sfballet.org" in url_l:
                    results["Type"] = "Ballet & Dance School"
                elif "linesballet.org" in url_l:
                    results["Type"] = "Professional Ballet Company & School"
                elif "sfcm.edu" in url_l:
                    results["Type"] = "Music Conservatory"
                elif "sfgirlschorus.org" in url_l:
                    results["Type"] = "Choral Music Organisation"
                elif "ypt.org" in url_l:
                    results["Type"] = "Youth Theatre Company"
                elif "wellsanfrancisco.com" in url_l:
                    results["Type"] = "Therapy & Wellness Clinic"
                elif "bayclubs.com" in url_l:
                    results["Type"] = "Fitness & Sports Club"
                elif "presidiogolf.com" in url_l:
                    results["Type"] = "Golf Course & Academy"
                elif "revolutionprep.com" in url_l:
                    results["Type"] = "Academic Test Prep & Tutoring"
                elif "crimsoneducation.org" in url_l:
                    results["Type"] = "College Admissions Consultancy"
                elif "spcs.stanford.edu" in url_l:
                    results["Type"] = "University Pre-Collegiate Programme"
                elif "atdp.berkeley.edu" in url_l:
                    results["Type"] = "University Academic Enrichment Programme"
                elif "lwhs.org" in url_l:
                    results["Type"] = "Independent Day School"
                elif "urbanschool.org" in url_l:
                    results["Type"] = "Independent Day School"
                elif "sfuhs.org" in url_l:
                    results["Type"] = "Independent High School"
                elif "hamlin.org" in url_l:
                    results["Type"] = "Independent All-Girls School"
                elif "townschool.com" in url_l:
                    results["Type"] = "Independent All-Boys School"
                elif "all-girls" in tl or "girls' school" in tl or "girls school" in tl:
                    results["Type"] = "Independent All-Girls School"
                elif "all-boys" in tl or "boys' school" in tl or "boys school" in tl:
                    results["Type"] = "Independent All-Boys School"
                elif "waldorf" in tl or "steiner" in tl:
                    results["Type"] = "Waldorf School"
                elif "montessori" in tl:
                    results["Type"] = "Montessori School"
                elif "ib world" in tl or "international baccalaureate" in tl:
                    results["Type"] = "IB World Day School"
                elif "college prep" in tl or "college preparatory" in tl:
                    results["Type"] = "College Preparatory School"
                elif "ballet" in tl and ("school" in tl or "academy" in tl):
                    results["Type"] = "Ballet & Dance School"
                elif "conservatory" in tl or "conservatoire" in tl:
                    results["Type"] = "Music Conservatory"
                elif "choir" in tl or "choral" in tl or "chorus" in tl:
                    results["Type"] = "Choral Music Organisation"
                elif "therapy" in tl or "counseling" in tl or "psychiatry" in tl:
                    results["Type"] = "Therapy & Wellness Clinic"
                elif "test prep" in tl or ("sat" in tl and "prep" in tl):
                    results["Type"] = "Academic Test Prep & Tutoring"
                else:
                    results["Type"] = "Independent School"

            soup_home = BeautifulSoup(await page.content(), "html.parser")
            for attr in [{"name": "description"}, {"property": "og:description"}]:
                tag = soup_home.find("meta", attr)
                if tag and tag.get("content"):
                    results["Tagline"] = tag["content"][:200]
                    break
            if not results["Tagline"]:
                results["Tagline"] = extract(r"([A-Z][^.]{30,150}\.)")

            results["Quote"] = scrape_quote_from_soup(soup_home)

            image_set = set()
            await harvest_images(page, url, image_set)
            results["Images"] = [img for img in image_set if img][:10]

        except Exception as e:
            print(f"Home page error: {e}")
            await browser.close()
            return None

        # ── SUBPAGE CRAWL ─────────────────────────────────────────────────────
        soup  = BeautifulSoup(await page.content(), "html.parser")
        links = soup.find_all("a", href=True)
        queue     = []
        seen_urls = {url.rstrip("/")}

        targets = {
            "about": ["about", "history", "welcome", "mission", "who-we-are",
                      "our-story", "overview", "vision", "values", "ethos"],
            "outcomes": ["results", "destination", "university", "leavers", "beyond",
                         "sixth-form", "exam", "college", "achievements", "academic-results",
                         "college-counseling", "college-placement"],
            "fees": ["fees", "tuition", "financial", "cost", "charges",
                     "fee-structure", "school-fees", "annual-fees", "pricing",
                     "affording", "financial-aid"],
            "admission": ["admissions", "apply", "entry", "register", "enroll",
                          "how-to-apply", "application", "join-us", "registration",
                          "enrolment", "admission-process"],
            "facilities": ["facilities", "campus", "co-curriculum", "sport", "arts",
                           "library", "infrastructure", "resources", "activities",
                           "co-curricular", "extracurricular", "programs", "academics"],
        }

        for link in links:
            href, text = link["href"], link.get_text().lower().strip()
            full_url   = urljoin(url, href).split("#")[0].rstrip("/")
            if full_url not in seen_urls and url.split("/")[2] in full_url:
                if any(k in text or k in href.lower() for cat in targets.values() for k in cat):
                    queue.append((full_url, text))
                    seen_urls.add(full_url)

        used_sentences: set = set()

        def _register(text: str):
            if not text:
                return
            for s in re.split(r'(?<=[.!?])\s+', text.strip()):
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and len(key) > 20:
                    used_sentences.add(key)

        def dedup_text(text: str) -> str:
            if not text:
                return text
            parts = re.split(r'(?<=[.!?])\s+', text.strip())
            fresh = []
            for s in parts:
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and key not in used_sentences and not is_junk(s):
                    fresh.append(s.strip())
                    used_sentences.add(key)
            return " ".join(fresh)

        def assign_field(field_name: str, text: str, max_sent: int = 2):
            if results.get(field_name):
                return
            cleaned = dedup_text(clean_to_sentences(text, max_sent))
            if cleaned:
                results[field_name] = cleaned

        for _field in ["About", "Tagline", "Philosophy", "Outcomes", "Admissions", "Quote"]:
            _register(results.get(_field, ""))

        used_paras_global: set = set()

        # Performance keywords — strict mapping to avoid cross-contamination
        PERF_KWS = {
            "Coaching Credentials":   [
                "qualified", "certified", "credential", "faculty", "expertise",
                "experienced teacher", "master", "phd", "trained", "professional development",
                "instructor background", "staff background",
            ],
            "Student Wellbeing":      [
                "wellbeing", "mental health", "counselor", "advisory", "pastoral",
                "social-emotional", "support program", "community", "belonging",
                "college counselor", "college counseling",
            ],
            "Academic Integration":   [
                "curriculum", "coursework", "program of study", "academic program",
                "core curriculum", "learning outcomes", "course offerings",
                "academic requirements", "subject", "ap course", "honors",
            ],
            "Competitive Pathway":    [
                "university placement", "college acceptance", "accepted to",
                "graduates attend", "alumni", "college counseling outcome",
                "ivy league", "top university", "competition winner", "national merit",
                "scholarship", "award winner",
            ],
            "Facilities & Resources": [
                "gymnasium", "theater", "theatre", "laboratory", "library",
                "swimming pool", "pool", "studio", "auditorium", "makerspace",
                "science lab", "art studio", "music room", "dance studio",
                "athletic field", "sports facility",
            ],
            "Ongoing Accountability": [
                "progress report", "parent conference", "student assessment",
                "feedback system", "portfolio", "transcript", "grade report",
                "parent communication", "accreditation", "wasc", "nais",
                "annual review", "learning plan",
            ],
        }

        for target_url, link_text in queue[:15]:
            try:
                await page.goto(target_url, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(2)
                main          = page.locator("main").first
                content_scope = main if await main.is_visible() else page
                paragraphs    = await content_scope.locator("p").all_text_contents()
                clean_paras   = [
                    t.strip() for t in paragraphs
                    if len(t.strip()) >= 60
                    and not any(x in t.lower() for x in ["cookie", "privacy", "menu", "navigation", "footer"])
                    and not is_junk(t)
                ]
                global_text_pool.extend(clean_paras)
                full_text = " ".join(clean_paras)
                page_text = await page.inner_text("body")
                sub_soup  = BeautifulSoup(await page.content(), "html.parser")

                await harvest_images(page, url, image_set)
                results["Images"] = [img for img in image_set if img][:10]

                if any(k in link_text or k in target_url for k in targets["about"]):
                    clean_about = [p for p in clean_paras if not is_junk(p)
                                   and not p.lower().startswith(("please fill", "please note",
                                   "the admissions team", "fill the", "click here", "select",
                                   "i'm excited", "i am excited", "one of the best parts",
                                   "we're glad", "we are glad"))]
                    assign_field("About", " ".join(clean_about), 2)
                    teach_kws  = ["curriculum", "teaching", "learning", "approach",
                                  "education", "ethos", "method", "philosophy", "program",
                                  "pedagogy", "inquiry", "project-based"]
                    teach_para = [p for p in clean_paras
                                  if any(k in p.lower() for k in teach_kws)
                                  and not is_junk(p)]
                    assign_field("Philosophy", " ".join(teach_para), 2)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)
                    if not results.get("Founded"):
                        sub_cands = extract_founded_from_text(page_text)
                        if sub_cands:
                            sub_cands.sort(key=lambda x: (-x[0], x[1]))
                            results["Founded"] = str(sub_cands[0][1])

                if not results.get("Students"):
                    for pat in student_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            num_str = m.group(1).replace(",", "")
                            try:
                                num = int(num_str)
                                if 50 <= num <= 10000 and not (2020 <= num <= 2030):
                                    results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                                    break
                            except ValueError:
                                pass

                if not results.get("Ratio"):
                    for pat in ratio_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            results["Ratio"] = m.group(1).replace(" ", "")
                            break

                if any(k in link_text or k in target_url for k in targets["outcomes"]):
                    # Outcomes: look for sentences about college/university destinations
                    outcome_kws = ["college", "university", "accepted", "acceptance",
                                   "placement", "graduate", "alumni", "destination",
                                   "enrolled at", "attend", "scholarship"]
                    outcome_paras = [p for p in clean_paras
                                     if any(k in p.lower() for k in outcome_kws)
                                     and not is_junk(p)]
                    assign_field("Outcomes", " ".join(outcome_paras) or full_text, 2)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)

                if any(k in link_text or k in target_url for k in targets["fees"]):
                    fee_val = extract_fee_validated(page_text)
                    if fee_val and not results.get("Fees"):
                        results["Fees"] = fee_val

                if any(k in link_text or k in target_url for k in targets["admission"]):
                    adm_kws  = ["apply", "application", "admission", "register",
                                "assessment", "entry", "enroll", "enrolment", "registration",
                                "open house", "inquiry", "shadow day"]
                    filtered = [p for p in clean_paras
                                if any(k in p.lower() for k in adm_kws)
                                and not is_junk(p)]
                    assign_field("Admissions", " ".join(filtered), 2)
                    dl = re.search(
                        r'(?:deadline|closes?|closing date)\s*[:\-]?\s*([^\n.]{10,60})',
                        page_text, re.I
                    )
                    if dl and results["AppDeadline"].startswith("Rolling"):
                        val = dl.group(1).strip()[:80]
                        has_month = re.search(
                            r'\b(january|february|march|april|may|june|july|august|'
                            r'september|october|november|december|jan|feb|mar|apr|'
                            r'jun|jul|aug|sep|oct|nov|dec)\b', val, re.I)
                        has_date = re.search(r'\b\d{1,2}[\/\-]\d{1,2}', val)
                        if (has_month or has_date) and not is_junk(val):
                            results["AppDeadline"] = val
                    if not results.get("Ages"):
                        for pat in age_patterns:
                            m = re.search(pat, page_text, re.I)
                            if m:
                                try:
                                    g = m.groups()
                                    if len(g) >= 2 and g[0] and g[1]:
                                        lo, hi = int(g[0]), int(g[1])
                                        if lo < hi and lo >= 2 and hi <= 25:
                                            results["Ages"] = f"{lo}–{hi}"
                                except Exception:
                                    pass
                                if results["Ages"]:
                                    break

                if any(k in link_text or k in target_url for k in targets["facilities"]):
                    for metric, kws in PERF_KWS.items():
                        if not results["Performance"].get(metric):
                            matched = [
                                p for p in clean_paras
                                if any(k in p.lower() for k in kws)
                                and p not in used_paras_global
                                and not is_junk(p)
                                and len(p.split()) >= 15  # must be substantive
                            ]
                            if matched:
                                chosen = matched[:1]
                                val    = dedup_text(clean_to_sentences(" ".join(chosen), 2))
                                if val and len(val.split()) >= 12:
                                    results["Performance"][metric] = val
                                    used_paras_global.update(chosen)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)

            except Exception as e:
                print(f"  Skipping {target_url}: {e}")
                continue

        results["Images"] = list(image_set)[:10]
        for field, fallback in FALLBACKS.items():
            if not results.get(field):
                results[field] = fallback
        if not results["City"]:
            results["City"] = "San Francisco, CA"

        if not results["Quote"]:
            print(f"  No quote found — generating AI quote…")
            results["Quote"] = generate_ai_quote(
                results["Name"], results["Type"], results["City"], results["About"]
            )

        await browser.close()

        # ── STEP 1: Fill main fields (with min-length enforcement for paragraphs)
        print(f"\n  Running OpenAI main field gap-fill…")
        results = openai_fill_missing_fields(results)

        # ── STEP 2: Force-fill ALL 6 performance fields (never blank/N/A/junk)
        print(f"\n  Running AI force-fill for performance fields…")
        results = openai_force_fill_performance(results)

        # ── POST SANITIZATION ─────────────────────────────────────────────────
        for field in ["Name", "Tagline", "Founded", "City", "Ages", "Students",
                      "Ratio", "Fees", "Type", "About", "Philosophy", "Outcomes",
                      "Admissions", "Quote"]:
            if results.get(field):
                results[field] = re.sub(r'\s+', ' ', str(results[field])).strip()
        for field in ["About", "Philosophy", "Outcomes", "Admissions"]:
            if results.get(field) and len(results[field].split()) > 60:
                results[field] = clean_to_sentences(results[field], 2)
        PERF_METRICS = ["Coaching Credentials", "Student Wellbeing", "Academic Integration",
                        "Competitive Pathway", "Facilities & Resources", "Ongoing Accountability"]
        for metric in PERF_METRICS:
            val = results["Performance"].get(metric, "")
            if val and len(val.split()) > 50:
                results["Performance"][metric] = clean_to_sentences(val, 2)

        # ── PRINT SUMMARY ─────────────────────────────────────────────────────
        SEP        = "—" * 55
        city_short = results["City"].split(",")[0]
        print(f"\n{SEP}")
        print(f"Elite › {city_short} › {results['Name']}")
        print(SEP)
        print(f"\n  Type       : {results['Type']}")
        print(f"  Ages       : {results['Ages'] or 'N/A'}")
        print(f"  Students   : {results['Students'] or 'N/A'}")
        print(f"  Ratio      : {results['Ratio'] or 'N/A'}")
        print(f"  Founded    : {results['Founded'] or 'N/A'}")
        print(f"  City       : {results['City']}")
        print(f"  Annual Fee : {results['Fees']}")
        print(f"  Images     : {len(results['Images'])} found")
        for i, img in enumerate(results["Images"], 1):
            print(f"    {i}. {img}")
        print(f"\n  About      : {results['About']}")
        print(f"\n  Quote      : \"{results['Quote']}\"")
        print(f"\n  Philosophy : {results['Philosophy']}")
        print(f"\n  Outcomes   : {results['Outcomes']}")
        print(f"\n  Admissions : {results['Admissions']}")
        print(f"\nPerformance Metrics:")
        for k, v in results["Performance"].items():
            kw = PERF_SHORT.get(k, k)
            print(f"\n  [{kw}] {k}:")
            print(f"  {v or '❌ STILL MISSING'}")
        print(f"\n  Website : {results['URL']}")
        print(SEP)

        # ── SAVE ──────────────────────────────────────────────────────────────
        wp_code     = generate_wp_blocks(results)
        school_slug = re.sub(r"[^a-z0-9]+", "_", results["Name"].lower())[:40].strip("_")
        out_wp      = f"{school_slug}_wp_blocks.txt"
        out_json    = f"{school_slug}_data.json"

        with open(out_wp, "w", encoding="utf-8") as f:
            f.write(wp_code)

        save_data               = {k: v for k, v in results.items() if k != "Images"}
        save_data["ImageCount"] = len(results.get("Images", []))
        save_data["ImageURLs"]  = results.get("Images", [])
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(save_data, f, ensure_ascii=False, indent=2)

        print(f"\n  WP blocks  → {out_wp}")
        print(f"  JSON data  → {out_json}")
        return results


# ── URLS ──────────────────────────────────────────────────────────────────────

school_urls = [
    "https://lwhs.org",
    "https://townschool.com",
    "https://urbanschool.org",
    "https://sfuhs.org",
    "https://hamlin.org",
    "https://bayclubs.com",
    "https://presidiogolf.com",
    "https://revolutionprep.com",
    "https://www.crimsoneducation.org/us",
    "https://spcs.stanford.edu/",
    "https://atdp.berkeley.edu",
    "https://sfballet.org/school",
    "https://sfcm.edu",
    "https://wellsanfrancisco.com/",
    "https://ypt.org",
    "https://sfgirlschorus.org",
    "https://linesballet.org",
]


async def run_multiple_schools():
    all_results = []
    for url in school_urls:
        print(f"\n{'=' * 60}")
        print(f"  Scraping: {url}")
        print(f"{'=' * 60}")
        try:
            result = await extract_school_data(url)
            if result:
                all_results.append(result)
            await asyncio.sleep(3)
        except Exception as e:
            print(f"  Failed: {url} → {e}")

    master_file = "all_schools_data_sanfrancisco.json"
    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n{'=' * 60}")
    print(f"All done! {len(all_results)} institutions scraped.")
    print(f"Master JSON → {master_file}")
    print(f"{'=' * 60}")
    return all_results


if __name__ == "__main__":
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future   = pool.submit(asyncio.run, run_multiple_schools())
            all_data = future.result()
    except RuntimeError:
        asyncio.run(run_multiple_schools())


  Scraping: https://lwhs.org
  ✓ Known institution: Lick-Wilmerding High School | City locked: San Francisco, CA
Connecting to: https://lwhs.org...
  No quote found — generating AI quote…
  ✓ AI quote generated for Lick-Wilmerding High School

  Running OpenAI main field gap-fill…
  → OpenAI filling 3 main field(s): Fees, Outcomes, Admissions
  ✓ OpenAI filled: Fees, Outcomes, Admissions

  Running AI force-fill for performance fields…
  → AI force-filling 4 performance field(s): Coaching Credentials, Competitive Pathway, Facilities & Resources, Ongoing Accountability
  ✓ AI filled performance: Coaching Credentials, Competitive Pathway, Facilities & Resources, Ongoing Accountability

———————————————————————————————————————————————————————
Elite › San Francisco › Lick-Wilmerding High School
———————————————————————————————————————————————————————

  Type       : Independent Day School
  Ages       : 14–18
  Students   : 550
  Ratio      : 8:1
  Founded    : 1895
  City       : San Franc